## Exercícios em SQL cobrindo:
- JOINs
- CTE
- Window Functions básicas

In [2]:
import pandas as pd
import sqlite3

### Banco de Dados (Loja Online)

In [3]:
connection = sqlite3.connect("./db/ecommerce.db")
crsr = connection.cursor()
print("Connected to the database")

Connected to the database


In [4]:
data_clientes = {
  'id_cliente': [1, 2, 3, 4],
  'nome': ['Ana Souza', 'Bruno Lima', 'Carla Mendes', 'Diego Alves'],
  'cidade': ['São Paulo', 'Rio de Janeiro', 'Belo Horizonte', 'São Paulo'],
  'data_cadastro': ['2023-01-10', '2023-02-15', '2023-03-20', '2023-04-05']
}

data_pedidos ={
  'id_pedido': [101, 102, 103, 104, 105, 106],
  'id_cliente': [1, 2, 1, 3, 2, 4],
  'data_pedido': ['2024-01-10', '2024-01-15', '2024-02-01', '2024-02-10', '2024-03-05', '2024-03-08'],
  'valor_total': [250.00, 180.00, 320.00, 150.00, 500.00, 90.00]
}

data_itens_pedido = {
  'id_item': [1, 2, 3, 4, 5, 6],
  'id_pedido': [101, 102, 103, 104, 105, 106],
  'produto': ['Teclado', 'Mouse', 'Monitor', 'Headset', 'Notebook', 'Mousepad'],
  'quantidade': [1, 2, 1, 1, 1, 3],
  'preco_unitario': [250.00, 90.00, 320.00, 150.00, 500.00, 30.00]
}

In [5]:
df_clientes = pd.DataFrame(data_clientes)
df_clientes.to_sql('clientes', connection, if_exists='replace', index=False)
print("DataFrame inserted into the database")

DataFrame inserted into the database


In [6]:
df_pedidos = pd.DataFrame(data_pedidos)
df_pedidos.to_sql('pedidos', connection, if_exists='replace', index=False)
print("DataFrame inserted into the database")

DataFrame inserted into the database


In [7]:
df_itens_pedido = pd.DataFrame(data_itens_pedido)
df_itens_pedido.to_sql('itens_pedido', connection, if_exists='replace', index=False)
print("DataFrame inserted into the database")

DataFrame inserted into the database


### Exercícios SQL

#### JOINs

In [8]:
#Liste o nome do cliente, data do pedido e valor_total de todos os pedidos.
sql_query = """ SELECT c.nome, p.data_pedido, p.valor_total
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente;"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,data_pedido,valor_total
0,Ana Souza,2024-01-10,250.0
1,Ana Souza,2024-02-01,320.0
2,Bruno Lima,2024-01-15,180.0
3,Bruno Lima,2024-03-05,500.0
4,Carla Mendes,2024-02-10,150.0


In [9]:
#Mostre todos os clientes, mesmo os que não fizeram pedido.
sql_query = """ SELECT c.nome, p.data_pedido, p.valor_total
FROM clientes c
LEFT JOIN pedidos p ON c.id_cliente = p.id_cliente;"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,data_pedido,valor_total
0,Ana Souza,2024-01-10,250.0
1,Ana Souza,2024-02-01,320.0
2,Bruno Lima,2024-01-15,180.0
3,Bruno Lima,2024-03-05,500.0
4,Carla Mendes,2024-02-10,150.0


In [10]:
#Liste os pedidos com seus produtos comprados.
sql_query = """ SELECT p.id_pedido, p.data_pedido, p.valor_total, i.produto
FROM itens_pedido i
JOIN pedidos p ON i.id_pedido = p.id_pedido;"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_pedido,data_pedido,valor_total,produto
0,101,2024-01-10,250.0,Teclado
1,102,2024-01-15,180.0,Mouse
2,103,2024-02-01,320.0,Monitor
3,104,2024-02-10,150.0,Headset
4,105,2024-03-05,500.0,Notebook


In [11]:
#Mostre clientes que fizeram pedidos acima de 300 reais.
sql_query = """SELECT c.nome, p.data_pedido, p.valor_total
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE p.valor_total > 300;"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,data_pedido,valor_total
0,Ana Souza,2024-02-01,320.0
1,Bruno Lima,2024-03-05,500.0


#### CTE

In [12]:
'''Crie uma CTE chamada total_por_cliente que mostre:

id_cliente
total_gasto

Depois selecione todos os dados dela.'''
sql_query = """WITH total_por_cliente AS(
SELECT c.id_cliente, SUM(p.valor_total) AS total_gasto
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente
)SELECT * FROM total_por_cliente
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_cliente,total_gasto
0,1,570.0
1,2,680.0
2,3,150.0
3,4,90.0


In [13]:
#Usando CTE, mostre o cliente que mais gastou.
sql_query = """WITH total_por_cliente AS(
SELECT c.id_cliente, c.nome, SUM(p.valor_total) AS total_gasto
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente
GROUP BY c.id_cliente, c.nome
)SELECT * FROM total_por_cliente ORDER BY total_gasto DESC LIMIT 1;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_cliente,nome,total_gasto
0,2,Bruno Lima,680.0


In [14]:
#Crie uma CTE com pedidos feitos em fevereiro de 2024 e depois selecione dela.
sql_query = """WITH pedidos_fevereiro AS(
SELECT c.nome,p.valor_total, p.data_pedido
FROM clientes c
JOIN pedidos p ON c.id_cliente = p.id_cliente
WHERE data_pedido >= '2024-02-01'
AND data_pedido < '2024-03-01'
)SELECT * FROM pedidos_fevereiro;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,nome,valor_total,data_pedido
0,Ana Souza,320.0,2024-02-01
1,Carla Mendes,150.0,2024-02-10


#### Window Functions Básicas

In [15]:
#Mostre todos os pedidos com uma coluna extra chamada ranking_valor, ranqueando do maior valor para o menor.
sql_query = """ SELECT *, RANK() OVER (ORDER BY valor_total DESC) AS ranking_valor FROM pedidos;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_pedido,id_cliente,data_pedido,valor_total,ranking_valor
0,105,2,2024-03-05,500.0,1
1,103,1,2024-02-01,320.0,2
2,101,1,2024-01-10,250.0,3
3,102,2,2024-01-15,180.0,4
4,104,3,2024-02-10,150.0,5


In [16]:
#Mostre cada pedido e o valor acumulado dos pedidos em ordem de data.
sql_query = """ SELECT *, SUM(valor_total) OVER ( ORDER BY data_pedido
 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS valor_acumulado FROM pedidos;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_pedido,id_cliente,data_pedido,valor_total,valor_acumulado
0,101,1,2024-01-10,250.0,250.0
1,102,2,2024-01-15,180.0,430.0
2,103,1,2024-02-01,320.0,750.0
3,104,3,2024-02-10,150.0,900.0
4,105,2,2024-03-05,500.0,1400.0


In [17]:
#Mostre o total gasto por cliente e uma coluna com a média geral de gasto entre todos os clientes.
sql_query = """ 
WITH total_cliente AS (
  SELECT id_cliente,
  SUM(valor_total) AS total_gasto
  FROM pedidos
  GROUP BY id_cliente
)
SELECT *, AVG(total_gasto) OVER () AS media_geral
FROM total_cliente;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_cliente,total_gasto,media_geral
0,1,570.0,372.5
1,2,680.0,372.5
2,3,150.0,372.5
3,4,90.0,372.5


In [18]:

#Para cada cliente, numere os pedidos do mais antigo para o mais recente
sql_query = """ SELECT *, ROW_NUMBER() OVER (PARTITION BY id_cliente ORDER BY data_pedido) AS numero_pedido FROM pedidos;
"""

df = pd.read_sql_query(sql_query, connection)
df.head(5)

,id_pedido,id_cliente,data_pedido,valor_total,numero_pedido
0,101,1,2024-01-10,250.0,1
1,103,1,2024-02-01,320.0,2
2,102,2,2024-01-15,180.0,1
3,105,2,2024-03-05,500.0,2
4,104,3,2024-02-10,150.0,1


In [19]:
connection.close()
print("Connection closed")

Connection closed
